# F5 — Expanded evaluation: is the refit adapter's gain real at rank 1?

The paired bootstrap on B3's 1,000-image eval set found the refit map's
R@5 and R@10 gains significant (95% CIs [+0.009, +0.035] and [+0.008,
+0.033]) but left R@1 undecided: +0.014 with an interval of [-0.006,
+0.033], from 55 queries gained against 41 lost. Rank 1 is volatile - a
single position swap flips it - so 1,000 queries is not enough to settle
a gain of that size.

**The fair-evaluation constraint.** `pairs.npz` holds 4,000 rows, but
3,000 of them fitted the shipped adapter, so only its 1,000 eval rows are
out-of-sample for both maps. Resampling those adds no information.

**The solution.** The distillation corpus used the first 40,000 sorted
train2017 ids; train2017 contains about 118,000 images. Everything beyond
the corpus is unseen by *both* maps - the shipped one (fitted on val2017)
and the refit one (fitted on the corpus). This notebook builds 5 disjoint
galleries of 1,000 from that untouched pool.

**Gallery size stays at 1,000**, because R@K depends on it and every
number in the report was measured at that size. Five galleries narrow the
interval by roughly sqrt(5) while keeping the metric comparable.

**Protocol matched to B3 exactly:** first caption per image as the query,
SigLIP text tower, padding to 64 tokens, cosine ranking, ground truth on
the diagonal.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
!pip -q install torch torchvision transformers open_clip_torch pillow

In [ ]:
import numpy as np, torch, json, io, zipfile, urllib.request, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

DATA_DIR = Path(os.environ["DATA_DIR"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
TEACHER_ID = "google/siglip2-base-patch16-224"   # must match F1 / B1
N_GALLERIES, GALLERY = 5, 1000
CKPT = DATA_DIR / "f5_eval_ckpt.npz"

d = np.load(str(DATA_DIR / "distill_corpus.npz"))
used = set(int(i) for i in d["ids"])
print(f"{len(used)} image ids are in the distillation corpus and excluded")

In [ ]:
import json, zipfile, urllib.request, io

# COCO annotations: cached in DATA_DIR, fetched only if absent or broken.
# ~250 MB zip. First run is the slowest cell; afterwards it is instant.
# You can also place the file in DATA_DIR yourself and it is used as-is.
ANN_URL = ("http://images.cocodataset.org/annotations/"
           "annotations_trainval2017.zip")
ANN = DATA_DIR / "annotations_trainval2017.zip"
MEMBER = "annotations/captions_train2017.json"

def _usable(path):
    """exists() is not enough - a truncated download passes it and then
    fails confusingly several cells later."""
    if not path.exists() or path.stat().st_size < 100_000_000:
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return MEMBER in z.namelist()
    except zipfile.BadZipFile:
        return False

if _usable(ANN):
    print(f"using cached annotations ({ANN.stat().st_size/1e6:.0f} MB)")
else:
    if ANN.exists():
        print("cached file is truncated or corrupt - re-downloading")
        ANN.unlink()
    print(f"downloading annotations to {ANN} (~250 MB, once) ...")
    tmp = ANN.with_suffix(".part")        # atomic: never leave a half file
    urllib.request.urlretrieve(ANN_URL, str(tmp))
    tmp.rename(ANN)
    assert _usable(ANN), "download completed but the archive is unreadable"
    print(f"done ({ANN.stat().st_size/1e6:.0f} MB)")

with zipfile.ZipFile(ANN) as z:
    with z.open(MEMBER) as f:
        ann = json.load(f)
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
url = {im["id"]: im["coco_url"] for im in ann["images"]}

pool = [i for i in sorted(set(caps) & set(url)) if i not in used]
need = N_GALLERIES * GALLERY
ids = pool[:int(need * 1.15)]          # headroom for failed downloads
print(f"{len(pool)} untouched images available; fetching {len(ids)}")
assert len(pool) >= need, "not enough unseen images - lower N_GALLERIES"

In [ ]:
from transformers import AutoProcessor, AutoModel
import open_clip

sp = AutoProcessor.from_pretrained(TEACHER_ID)
sm = AutoModel.from_pretrained(TEACHER_ID).to(DEV).eval()
mob_model, _, mob_pre = open_clip.create_model_and_transforms(
    "MobileCLIP-S1", pretrained="datacompdr")
mob_model = mob_model.to(DEV).eval()

def _t(o):
    if torch.is_tensor(o): return o
    for a in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(o, a, None)
        if v is not None: return v
    return o.last_hidden_state.mean(1)

def fetch(i):
    try:
        with urllib.request.urlopen(url[i], timeout=8) as r:
            return i, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return i, None

if CKPT.exists():
    z = np.load(str(CKPT))
    MOB, SIG, TXT, KEEP = ([z["mob"]], [z["sig"]], [z["txt"]],
                           list(z["ids"]))
    start = int(z["next"])
    print(f"resuming at {start}")
else:
    MOB, SIG, TXT, KEEP, start = [], [], [], [], 0

t0 = time.time()
with ThreadPoolExecutor(max_workers=32) as ex:
    for c0 in range(start, len(ids), 512):
        c1 = min(c0 + 512, len(ids))
        got = [(i, im) for i, im in ex.map(fetch, ids[c0:c1]) if im is not None]
        for b in range(0, len(got), 32):
            chunk = got[b:b + 32]
            ims = [im for _, im in chunk]
            txts = [caps[i][0] for i, _ in chunk]        # FIRST caption
            with torch.no_grad():
                x = sp(images=ims, return_tensors="pt").to(DEV)
                y = sp(text=txts, padding="max_length", max_length=64,
                       truncation=True, return_tensors="pt").to(DEV)
                try:
                    si = _t(sm.get_image_features(**x))
                    st = _t(sm.get_text_features(**y))
                except Exception:
                    out = sm(input_ids=y["input_ids"],
                             pixel_values=x["pixel_values"])
                    si, st = out.image_embeds, out.text_embeds
                mv = mob_model.encode_image(
                    torch.stack([mob_pre(im) for im in ims]).to(DEV))
            SIG.append(si.float().cpu().numpy())
            TXT.append(st.float().cpu().numpy())
            MOB.append(mv.float().cpu().numpy())
            KEEP += [i for i, _ in chunk]
        r = (c1 - start) / max(time.time() - t0, 1e-9)
        print(f"  {c1}/{len(ids)} kept {len(KEEP)} {r:.0f} img/s "
              f"ETA {(len(ids)-c1)/max(r,1e-9)/60:.1f} min")
        np.savez_compressed(str(CKPT),
                            mob=np.concatenate(MOB).astype(np.float32),
                            sig=np.concatenate(SIG).astype(np.float32),
                            txt=np.concatenate(TXT).astype(np.float32),
                            ids=np.array(KEEP), next=c1)

MOB = np.concatenate(MOB).astype(np.float64)
SIG = np.concatenate(SIG).astype(np.float64)
TXT = np.concatenate(TXT).astype(np.float64)
print(f"\nfresh eval set: {len(MOB)} images, none seen by either map")

## The two maps, scored on 5 disjoint galleries

In [ ]:
def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
W_ship = ad["W_ridge"].astype(np.float64)

tr = d["train_idx"]
X = d["mob"][tr].astype(np.float64); Y = d["sig"][tr].astype(np.float64)
W_big = np.linalg.solve(X.T @ X + 1e-2 * np.eye(X.shape[1]), X.T @ Y)
print(f"shipped W fitted on {len(ad['train_idx'])} rows; "
      f"refit W on {len(tr)} rows")

n_use = min(len(MOB), N_GALLERIES * GALLERY)
hits = {name: {k: [] for k in (1, 5, 10)}
        for name in ("ceiling", "shipped", "refit")}
per_gallery = {name: [] for name in hits}

for g in range(n_use // GALLERY):
    sl = slice(g * GALLERY, (g + 1) * GALLERY)
    q = l2n(TXT[sl]); idx = np.arange(GALLERY)
    gals = {"ceiling": l2n(SIG[sl]),
            "shipped": l2n(MOB[sl] @ W_ship),
            "refit":   l2n(MOB[sl] @ W_big)}
    row = {}
    for name, G in gals.items():
        rank = (np.argsort(-(q @ G.T), 1) == idx[:, None]).argmax(1)
        for k in (1, 5, 10):
            hits[name][k].append(rank < k)
        row[name] = {k: float((rank < k).mean()) for k in (1, 5, 10)}
    for name in gals:
        per_gallery[name].append(row[name])
    print(f"gallery {g+1}: " + "   ".join(
        f"{n} R@1={row[n][1]:.3f}" for n in ("ceiling","shipped","refit")))

print(f"\n{'variant':10s} " + "  ".join(f"{'R@'+str(k):>14s}"
                                         for k in (1,5,10)))
for name in ("ceiling", "shipped", "refit"):
    cells = []
    for k in (1, 5, 10):
        v = [r[k] for r in per_gallery[name]]
        cells.append(f"{np.mean(v):.3f} ± {np.std(v):.3f}")
    print(f"{name:10s} " + "  ".join(f"{c:>14s}" for c in cells))

## The decisive test: paired bootstrap on all pooled queries

In [ ]:
rng = np.random.default_rng(0)
print(f"pooled over {n_use // GALLERY} galleries "
      f"({(n_use // GALLERY) * GALLERY} queries)\n")
print(f"{'K':4s} {'shipped':>8s} {'refit':>8s} {'gain':>8s} "
      f"{'gained':>7s} {'lost':>6s}  {'95% CI':>22s}   verdict")
for k in (1, 5, 10):
    a = np.concatenate(hits["shipped"][k])
    b = np.concatenate(hits["refit"][k])
    n = len(a)
    gained, lost = int((~a & b).sum()), int((a & ~b).sum())
    boot = np.array([ (b[s].mean() - a[s].mean())
                      for s in (rng.integers(0, n, n) for _ in range(5000)) ])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    v = "REAL" if lo > 0 else ("REGRESSION" if hi < 0 else "within noise")
    print(f"R@{k:<2d} {a.mean():8.3f} {b.mean():8.3f} "
          f"{b.mean()-a.mean():+8.4f} {gained:7d} {lost:6d}  "
          f"[{lo:+.4f}, {hi:+.4f}]   {v}")

c1 = np.concatenate(hits["ceiling"][1]).mean()
r1 = np.concatenate(hits["refit"][1]).mean()
s1 = np.concatenate(hits["shipped"][1]).mean()
print(f"\nR@1 as % of ceiling: shipped {100*s1/c1:.1f}%   "
      f"refit {100*r1/c1:.1f}%   (pre-registered gate: 90%)")

## Reading the result

The interval, not the point estimate, decides. If R@1's 95% interval now
excludes zero, the gain is real and the refit adapter can be reported as
meeting the gate that Section 4.3 records as missed. If it still contains
zero at five times the sample size, the honest conclusion is that the
rank-1 gain is small enough to be indistinguishable at this scale, and
the claim rests on R@5 and R@10 — which are the metrics the product
surface actually uses.

Note that these galleries are drawn from train2017 while the report's
figures come from val2017, so absolute values may shift slightly. The
comparison that matters is *between the two maps on identical queries*,
which is unaffected.